# 04 · Linear Algebra Essentials

The hands-on companion to [`index.html`](index.html) in this folder. Read a lesson part on the
page first, then run the matching section here.

This notebook assumes **Modules 01–03 and nothing more**. You have never used NumPy before; by the
end you will have implemented the dot product, the norm, matrix–vector multiplication, and matrix
multiplication from scratch in plain Python, checked each against NumPy, and built a working image
classifier out of nothing but a dot product.

By the end you will be able to:

- create vectors and matrices in NumPy and read their shapes without guessing;
- implement `dot`, `norm`, `matvec`, and `matmul` from scratch and verify them against NumPy;
- compute distances and cosine similarities, and say which one a problem needs;
- apply a matrix to a shape and see what the transformation does;
- read a determinant and a rank, and recognise a singular matrix;
- solve `Ax = b`, and fit a least-squares line three different ways that all agree;
- find eigenvectors and run a small PCA; and
- classify handwritten digits with a single dot product.

**Working rule, unchanged from Module 02: predict first, run second, explain third.** A wrong
prediction is useful — it shows exactly which mental model needs adjusting.

## 0 · How to use this notebook

Click a code cell and press **Shift + Enter** to run it and move to the next one. Run it top to
bottom the first time. If it starts behaving strangely because cells were run out of order, use
**Kernel → Restart Kernel and Run All Cells**.

Cells marked **Predict before running** contain a comment line for your guess. Fill it in before you
run the cell. That habit is the single fastest way to learn this material.

Exercises are marked `TODO`. Each one is followed by a `check(...)` call that tells you `PASS` or
`FAIL` immediately — you do not need to wait for anyone to mark it. Full solutions are in section 12
at the end, but use them to check your work rather than to avoid it.

### Companion map

| Lesson part on the page | Section here |
|---|---|
| Part 1 · Data is a grid of numbers | 1 |
| Part 2 · Adding and scaling | 2 |
| Part 3 · Length and distance | 3 |
| Part 4 · The dot product | 4 |
| Part 5 · Matrices as functions | 5 |
| Part 6 · Matrix × matrix | 6 |
| Part 7 · Determinant, rank, inverse | 7 |
| Part 8 · Solving Ax = b | 8 |
| Part 9 · Eigenvectors and PCA | 9 |
| Part 10 · Reference and self-check | 10, 11 |

Everything here uses NumPy, Matplotlib, and (in section 10) scikit-learn's built-in digits dataset.
Nothing is downloaded and nothing is written to disk.

In [ ]:
# Setup. Run this before anything else.
import sys
import pathlib

sys.path.append(str(pathlib.Path('../../shared/notebook_utils').resolve()))
from course_utils import set_style, rng

import numpy as np
import matplotlib.pyplot as plt

set_style()
np.set_printoptions(precision=3, suppress=True)   # readable output: 3 decimals, no scientific notation

print('numpy version     :', np.__version__)
print('setup complete — everything below is arithmetic you can also do on paper')

In [ ]:
def check(name, actual, expected, tol=1e-9):
    '''Print PASS/FAIL for one expected value. Works for numbers, lists, and arrays.'''
    if actual is None:
        print(f'--    {name}: not attempted yet')
        return
    try:
        got = np.asarray(actual, dtype=float)
        want = np.asarray(expected, dtype=float)
    except (TypeError, ValueError):
        print(f'PASS  {name}' if actual == expected else f'FAIL  {name}: got {actual!r}, expected {expected!r}')
        return
    if got.shape != want.shape:
        print(f'FAIL  {name}: shape {got.shape}, expected shape {want.shape}')
    elif np.allclose(got, want, atol=tol, rtol=0):
        print(f'PASS  {name}')
    else:
        print(f'FAIL  {name}: got {got}, expected {want}')


def check_shape(name, array, expected_shape):
    '''Print PASS/FAIL for an array's shape alone.'''
    if array is None:
        print(f'--    {name}: not attempted yet')
    elif np.shape(array) == expected_shape:
        print(f'PASS  {name}  (shape {expected_shape})')
    else:
        print(f'FAIL  {name}: shape {np.shape(array)}, expected {expected_shape}')


def show_error(func, *args, **kwargs):
    '''Call func and print the exception instead of crashing the notebook.

    Use it to meet an error on purpose — shape errors are worth reading, not avoiding.
    '''
    try:
        result = func(*args, **kwargs)
    except Exception as error:
        print(f'{type(error).__name__}: {error}')
        return None
    print(f'no exception — the call returned:\n{result}')
    return result


# Self-tests, so you know what each outcome looks like.
check('check · a match', 4, 4)
check('check · an array match', [1.0, 2.0], [1, 2])
check('check · not attempted', None, 4)
check('check · a miss', 5, 4)
check('check · wrong shape', [[1, 2]], [1, 2])
print()
check_shape('check_shape · correct', np.zeros((3, 2)), (3, 2))

## 1 · Vectors, arrays, and shape

*Companion to [Part 1 · Data is a grid of numbers](index.html#why).*

A vector is a list of numbers. A matrix is a grid. In NumPy both are `ndarray` objects, and the
`shape` attribute is what tells them apart — reading it correctly is the single most useful
debugging habit in this whole course.

In [ ]:
# The five flowers from Demo 1 on the lesson page.
flower_a = np.array([1.4, 0.2])          # petal length, petal width
flower_b = np.array([4.7, 1.4])

print('flower_a       :', flower_a)
print('type           :', type(flower_a))
print('shape          :', flower_a.shape)     # (2,) — one axis, two entries
print('ndim (axes)    :', flower_a.ndim)      # 1 — it is a vector
print('first entry    :', flower_a[0])        # Python counts from 0; the textbook would call this v1
print('last entry     :', flower_a[-1])

`shape` is `(2,)` — note the trailing comma. That is NumPy's way of writing "one axis, of length 2".
It is **not** the same as `(2, 1)`, which would be a matrix with two rows and one column. This
distinction causes real bugs and we will meet it deliberately in a moment.

In [ ]:
# The whole dataset as one matrix: five rows (examples), two columns (features).
X = np.array([
    [1.4, 0.2],     # A
    [4.7, 1.4],     # B
    [5.1, 1.9],     # C
    [1.3, 0.4],     # D
    [4.5, 1.5],     # E
])

print('X =')
print(X)
print()
print('shape           :', X.shape)          # (5, 2) — ROWS FIRST, ALWAYS
print('number of rows  :', X.shape[0], '(examples)')
print('number of cols  :', X.shape[1], '(features)')
print('X[0]            :', X[0], '  <- row 0: everything about flower A')
print('X[:, 0]         :', X[:, 0], '  <- column 0: petal length for every flower')
print('X[1, 0]         :', X[1, 0], '  <- row 1, column 0. Row index first.')

### Predict before running · what does the transpose do?

`X.T` flips the matrix across its diagonal. Before you run the next cell, write down what shape you
expect, and what one row of `X.T` will contain.

In [ ]:
# My prediction: X.T has shape ______ and one row of it holds ________________________

print('X.shape   :', X.shape)
print('X.T.shape :', X.T.shape)
print()
print('X.T =')
print(X.T)
print()
print('X.T[0] :', X.T[0], '  <- every petal LENGTH, across all five flowers')

Transposing swaps the question from "tell me everything about this flower" to "tell me about this
measurement across all flowers". The shape numbers simply swap: `(5, 2)` becomes `(2, 5)`.

### The `(n,)` versus `(n, 1)` trap

This is worth meeting now, on purpose, in a controlled setting — because when it bites later it does
so silently.

In [ ]:
flat = np.array([1, 2, 3])            # shape (3,)  — a vector
column = flat.reshape(-1, 1)          # shape (3, 1) — a matrix with one column
row_form = flat.reshape(1, -1)        # shape (1, 3) — a matrix with one row

print('flat    ', flat.shape, flat)
print('column  ', column.shape); print(column)
print('row_form', row_form.shape, row_form)
print()
print('Now the dangerous part — adding a column to a row:')
print((column + row_form).shape, ' <- NOT (3,) and NOT an error!')
print(column + row_form)
print()
print('NumPy BROADCAST them into a 3x3 grid of every pairwise sum.')
print('No exception was raised. This is a wrong answer that looks like a real one,')
print('which is why you print .shape whenever a result surprises you.')

### Exercise 1 · shapes

Build the objects described in the comments. `check_shape` will tell you whether each is right.

In [ ]:
# TODO 1a: a vector holding the numbers 5, 0, -2, 7  (one axis)
vec = None

# TODO 1b: a 3x2 matrix whose rows are (2, 5), (0, 1) and (-3, 4)
mat = None

# TODO 1c: the transpose of `mat`
mat_t = None

# TODO 1d: from `vec`, a version with shape (4, 1) -- use .reshape
vec_column = None

check_shape('1a · vec', vec, (4,))
check('1a · contents', vec, [5, 0, -2, 7])
check_shape('1b · mat', mat, (3, 2))
check('1b · contents', mat, [[2, 5], [0, 1], [-3, 4]])
check_shape('1c · mat_t', mat_t, (2, 3))
check_shape('1d · vec_column', vec_column, (4, 1))

### Exercise 2 · reading a data matrix

A dataset holds 200 songs with 4 features each. Answer with code, not by hand.

In [ ]:
songs = rng(0).normal(size=(200, 4))       # a stand-in dataset with the right shape

# TODO 2a: how many examples are there? (read it from .shape, do not hard-code 200)
n_examples = None

# TODO 2b: how many features?
n_features = None

# TODO 2c: the 8th song's 3rd feature. Careful: Python counts from 0.
value = None

# TODO 2d: the mean of every feature, as a vector of length 4.
#          Hint: songs.mean(axis=0) averages DOWN the rows, giving one number per column.
feature_means = None

check('2a · n_examples', n_examples, 200)
check('2b · n_features', n_features, 4)
check('2c · value', value, songs[7, 2])
check_shape('2d · feature_means', feature_means, (4,))
check('2d · values', feature_means, songs.mean(axis=0))

## 2 · Vector arithmetic and linear combinations

*Companion to [Part 2 · Vectors: adding, scaling, and what you can reach](index.html#vector-ops).*

Two operations, and everything else is built from them: **add** and **scale**.

In [ ]:
u = np.array([3, 1])
v = np.array([1, 2])

print('u        =', u)
print('v        =', v)
print('u + v    =', u + v, '   <- component-wise')
print('v + u    =', v + u, '   <- the same: order does not matter for addition')
print('u - v    =', u - v, '   <- the arrow pointing FROM v TO u')
print('v - u    =', v - u, '   <- the opposite arrow')
print('2 * u    =', 2 * u, '   <- scaled: twice as long, same direction')
print('-1 * u   =', -1 * u, '  <- flipped: same line, opposite direction')
print('0 * u    =', 0 * u, '   <- the zero vector: no direction at all')

In [ ]:
# Draw it: the parallelogram from Demo 2.
fig, ax = plt.subplots(figsize=(5.5, 5.5))
origin = np.zeros(2)
total = u + v

for vec, color, label in [(u, '#2a78d6', 'u'), (v, '#1baf7a', 'v'), (total, '#eb6834', 'u+v')]:
    ax.annotate('', xy=vec, xytext=origin, arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(vec[0] + 0.12, vec[1] + 0.12, label, color=color, fontsize=13, fontweight='bold')

# The two walking routes, tip to tail.
ax.plot([u[0], total[0]], [u[1], total[1]], '--', color='gray', lw=1)
ax.plot([v[0], total[0]], [v[1], total[1]], '--', color='gray', lw=1)

ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 4); ax.set_aspect('equal')
ax.set_title('u + v is the fourth corner of the parallelogram')
plt.show()

### Predict before running · linear combinations and span

`u = (2, 1)` and `w = (-4, -2)`. Before running: can `a*u + b*w` ever reach the point `(0, 3)`?
Write your reasoning in the comment.

In [ ]:
# My prediction: ______ , because ____________________________________________

u2 = np.array([2, 1])
w2 = np.array([-4, -2])

print('Is w2 a multiple of u2?  w2 / u2 =', w2 / u2)
print()
print('A few combinations a*u2 + b*w2:')
for a, b in [(1, 0), (0, 1), (1, 1), (2, -1), (3, 0.5), (-1, 2)]:
    result = a * u2 + b * w2
    print(f'  a={a:>4}, b={b:>4}  ->  {result}   ratio y/x = {result[1] / result[0] if result[0] else float("nan"):.3f}')
print()
print('Every result has y/x = 0.5 -- they all lie on the SAME LINE through the origin.')
print('(0, 3) is not on that line, so no combination can ever reach it.')
print('w2 = -2 * u2, so it is redundant: the two vectors are linearly DEPENDENT.')

### Exercises 3–5

In [ ]:
# TODO 3: compute 3*(2, -1) - 2*(1, 4) using NumPy. Do it on paper too and compare.
result_3 = None
check('3 · combination', result_3, [4, -11])

# TODO 4: find the arrow that points FROM the point (1, 5) TO the point (4, 1).
#         Remember: destination minus start.
arrow_4 = None
check('4 · arrow from (1,5) to (4,1)', arrow_4, [3, -4])

# TODO 5: write a function that builds a linear combination of two vectors.
#         It should return a * vec1 + b * vec2.
def combine(a, vec1, b, vec2):
    pass   # replace with your implementation

if combine(2, np.array([1, 0]), 3, np.array([1, 1])) is not None:
    check('5 · combine', combine(2, np.array([1, 0]), 3, np.array([1, 1])), [5, 3])
    check('5 · combine again', combine(-1, np.array([2, 2]), 0.5, np.array([4, 0])), [0, -2])
else:
    print('--    5 · combine: not attempted yet')

## 3 · Norms, unit vectors, distance

*Companion to [Part 3 · Length, distance, and direction](index.html#norms).*

First implement the norm yourself, from Pythagoras, then check it against NumPy. Writing the loop
once is what stops `np.linalg.norm` being magic.

In [ ]:
def norm_from_scratch(vec):
    '''The L2 norm: the square root of the sum of squares. Pythagoras, generalised.'''
    total = 0.0
    for value in vec:
        total += value ** 2          # squaring makes every term positive
    return total ** 0.5              # the square root brings us back to the original units


test = np.array([3, 4])
print('my norm    :', norm_from_scratch(test))
print('numpy norm :', np.linalg.norm(test))
print('match      :', np.isclose(norm_from_scratch(test), np.linalg.norm(test)))
print()

# It works unchanged in any number of dimensions -- the picture stops at 3, the arithmetic never does.
four_d = np.array([1, 2, 2, 4])
print('4-D vector :', four_d)
print('its length :', norm_from_scratch(four_d), '(check: 1 + 4 + 4 + 16 = 25, and sqrt(25) = 5)')

In [ ]:
# Distance is the length of the difference. Subtract FIRST, then measure.
a = np.array([1.4, 0.2])       # flower A
b = np.array([4.7, 1.4])       # flower B
d = np.array([1.3, 0.4])       # flower D

print('A to B :', np.linalg.norm(b - a))
print('A to D :', np.linalg.norm(d - a))
print('B to A :', np.linalg.norm(a - b), ' <- same as A to B: distance is symmetric')
print()
print('A and D are close together; B is far away. That is the clustering you did by eye in Demo 1,')
print('and it is all that k-nearest-neighbours (Module 18) ever computes.')

In [ ]:
# Unit vectors: keep the direction, discard the length.
v = np.array([3, 4])
length = np.linalg.norm(v)
unit = v / length

print('v          :', v)
print('length     :', length)
print('unit vector:', unit)
print('its length :', np.linalg.norm(unit), ' <- 1, by construction')
print()

# The trap: normalising the zero vector.
zero = np.array([0.0, 0.0])
with np.errstate(invalid='ignore'):          # suppress the warning so we can see the result
    print('zero / its norm =', zero / np.linalg.norm(zero))
print('nan -- not a crash. It then spreads silently through every later calculation.')
print('Guard it:', zero / np.linalg.norm(zero) if np.linalg.norm(zero) > 0 else 'norm is zero, skipping')

In [ ]:
# L1 versus L2, and why the gap grows with dimension.
print(f'{"vector":<28}{"L2":>8}{"L1":>8}')
for vec in [np.array([3, 4]), np.array([1, 1, 1, 1]), np.ones(9)]:
    l2 = np.linalg.norm(vec)          # default is L2
    l1 = np.linalg.norm(vec, 1)       # pass 1 for the L1 norm
    print(f'{str(vec):<28}{l2:>8.3f}{l1:>8.3f}')
print()
print('L1 grows linearly with the number of entries; L2 grows with the square root.')
print('That widening gap is one face of the curse of dimensionality (Module 18).')

### Exercises 6–8

In [ ]:
# TODO 6: compute the L2 norm of (5, 12) WITHOUT np.linalg.norm.
#         Use your norm_from_scratch, or arithmetic.
norm_6 = None
check('6 · norm of (5,12)', norm_6, 13.0)

# TODO 7: write a function returning the distance between two points.
def distance(p, q):
    pass   # replace with your implementation

if distance(np.array([1, 1]), np.array([4, 5])) is not None:
    check('7 · distance', distance(np.array([1, 1]), np.array([4, 5])), 5.0)
    check('7 · symmetry', distance(np.array([4, 5]), np.array([1, 1])), 5.0)
else:
    print('--    7 · distance: not attempted yet')

# TODO 8: write a function that normalises a vector to length 1,
#         and returns the vector unchanged if its norm is 0 (do not produce nan).
def normalise(vec):
    pass   # replace with your implementation

if normalise(np.array([0.0, -2.0])) is not None:
    check('8 · normalise', normalise(np.array([0.0, -2.0])), [0.0, -1.0])
    check('8 · handles zero', normalise(np.array([0.0, 0.0])), [0.0, 0.0])
    check('8 · result has length 1', np.linalg.norm(normalise(np.array([3.0, 4.0]))), 1.0)
else:
    print('--    8 · normalise: not attempted yet')

## 4 · Dot products, angles, cosine similarity

*Companion to [Part 4 · The dot product](index.html#dot).*

The most useful operation in machine learning. Implement it, then use it three ways.

In [ ]:
def dot_from_scratch(a, b):
    '''Multiply matching entries, add up the products. Returns one number.'''
    if len(a) != len(b):
        raise ValueError(f'vectors must be the same length, got {len(a)} and {len(b)}')
    total = 0.0
    for ai, bi in zip(a, b):
        total += ai * bi
    return total


u = np.array([3, 1])
v = np.array([1, 2])

print('my dot     :', dot_from_scratch(u, v))
print('numpy @    :', u @ v)                 # the modern, readable way
print('np.dot     :', np.dot(u, v))          # the same thing
print()
print('Compare with ELEMENT-WISE multiplication:')
print('u * v      :', u * v, '  <- a vector, NOT the dot product')
print('sum(u * v) :', np.sum(u * v), '     <- summing it gives the dot product back')
print()
print('Mnemonic: @ collapses to one number, * preserves the shape.')

In [ ]:
# The sign of the dot product is the message.
fixed = np.array([3, 0])
print(f'{"other vector":<18}{"dot":>8}   meaning')
for other, meaning in [
    (np.array([2, 0]),  'same direction'),
    (np.array([2, 2]),  'less than 90 degrees apart'),
    (np.array([0, 3]),  'exactly perpendicular (orthogonal)'),
    (np.array([-1, 2]), 'more than 90 degrees apart'),
    (np.array([-3, 0]), 'exactly opposite'),
]:
    print(f'{str(other):<18}{fixed @ other:>8.1f}   {meaning}')

In [ ]:
# The two definitions of the dot product, computed separately, always agree.
def cosine_similarity(a, b):
    '''Cosine similarity for two nonzero vectors; always between -1 and 1.'''
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        raise ValueError('cosine similarity is undefined for a zero vector')
    return (a @ b) / denominator


u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])

algebraic = u @ v
cos_theta = cosine_similarity(u, v)
geometric = np.linalg.norm(u) * np.linalg.norm(v) * cos_theta
angle_degrees = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))

print(f'algebraic   u . v            = {algebraic:.6f}')
print(f'geometric   |u| |v| cos(t)   = {geometric:.6f}')
print(f'they agree                   : {np.isclose(algebraic, geometric)}')
print()
print(f'cos(theta)                   = {cos_theta:.4f}')
print(f'the angle between them       = {angle_degrees:.2f} degrees')

In [ ]:
# Why cosine, not distance, for documents.
# Each vector is (how often it says "model", how often it says "recipe").
short_ml = np.array([8.0, 1.0])       # a short article about ML
long_ml = np.array([80.0, 10.0])      # the SAME article, ten times longer
cooking = np.array([1.0, 7.0])        # an article about cooking

print(f'{"pair":<28}{"distance":>10}{"cosine":>10}')
for name, p, q in [
    ('short_ml vs long_ml', short_ml, long_ml),
    ('short_ml vs cooking', short_ml, cooking),
    ('long_ml  vs cooking', long_ml, cooking),
]:
    print(f'{name:<28}{np.linalg.norm(p - q):>10.2f}{cosine_similarity(p, q):>10.3f}')
print()
print('By DISTANCE, the two ML articles look further apart than short_ml and cooking.')
print('By COSINE, they are identical (1.000) -- because length is not topic.')
print('This is exactly why embedding search ranks by cosine similarity (Module 42).')

### Predict before running · a prediction is a dot product

A trained model has weights `w = (0.2, 15, -1.5)` and bias `b = 50`, for features
`(square feet, bedrooms, age in years)`. Before running: what does the *negative* weight on age
mean, in plain English?

In [ ]:
# My prediction: the negative weight on age means _______________________________

w = np.array([0.2, 15.0, -1.5])
b = 50.0
house = np.array([1500.0, 3.0, 12.0])

prediction = w @ house + b
print('features   :', house)
print('weights    :', w)
print('w . x      :', w @ house)
print('+ bias     :', b)
print('prediction :', prediction)
print()
print('Contribution of each feature:')
for name, weight, value in zip(['sq ft', 'bedrooms', 'age'], w, house):
    print(f'  {name:<10} {weight:>7.2f} x {value:>7.1f} = {weight * value:>9.1f}')
print()
print('The weight signs are the most readable thing about a linear model:')
print('holding size and bedrooms fixed, each extra year of age lowers the prediction.')

### Exercises 9–12

In [ ]:
# TODO 9: compute (1, 2, 3) . (4, 5, 6) using @.
dot_9 = None
check('9 · dot product', dot_9, 32)

# TODO 10: find ANY nonzero vector orthogonal to (3, 4).
#          Hint: the swap-and-negate trick, (a, b) -> (-b, a), always works in 2-D.
orthogonal_10 = None
if orthogonal_10 is not None:
    check('10 · dot product is zero', np.array(orthogonal_10) @ np.array([3, 4]), 0.0)
    check('10 · and it is nonzero', float(np.linalg.norm(orthogonal_10) > 0), 1.0)
else:
    print('--    10 · orthogonal: not attempted yet')

# TODO 11: compute the cosine similarity of (1, 0) and (1, 1) using your own arithmetic
#          (dot product divided by the two norms). It should be about 0.7071.
cos_11 = None
check('11 · cosine similarity', cos_11, 0.7071067811865475, tol=1e-6)

# TODO 12: write a function returning the PROJECTION VECTOR of a onto b:
#              proj = ((a . b) / (b . b)) * b
def project(a, b):
    pass   # replace with your implementation

if project(np.array([4.0, 3.0]), np.array([1.0, 0.0])) is not None:
    check('12 · project onto the x-axis', project(np.array([4.0, 3.0]), np.array([1.0, 0.0])), [4.0, 0.0])
    check('12 · project onto itself', project(np.array([2.0, 2.0]), np.array([2.0, 2.0])), [2.0, 2.0])
else:
    print('--    12 · project: not attempted yet')

## 5 · Matrices, shape, and matrix–vector products

*Companion to [Part 5 · Matrices as functions on space](index.html#matrices).*

Now the two readings of `Ax`. Implement both, and confirm they give the same answer.

In [ ]:
def matvec_rows(A, x):
    '''The ROW view: each output entry is one row of A dotted with x.'''
    return np.array([dot_from_scratch(row, x) for row in A])


def matvec_columns(A, x):
    '''The COLUMN view: the entries of x say how much of each column to take.'''
    total = np.zeros(A.shape[0])
    for j in range(A.shape[1]):
        total = total + x[j] * A[:, j]      # x[j] lots of column j
    return total


A = np.array([[2.0, 0.0],
              [1.0, 3.0]])
x = np.array([2.0, 1.0])

print('row view    :', matvec_rows(A, x))
print('column view :', matvec_columns(A, x))
print('numpy A @ x :', A @ x)
print('all three agree:', np.allclose(matvec_rows(A, x), A @ x) and np.allclose(matvec_columns(A, x), A @ x))
print()
print('Column view, written out:')
print(f'  x[0] * column 0 = {x[0]} * {A[:, 0]} = {x[0] * A[:, 0]}')
print(f'  x[1] * column 1 = {x[1]} * {A[:, 1]} = {x[1] * A[:, 1]}')
print(f'  sum             = {x[0] * A[:, 0] + x[1] * A[:, 1]}')

In [ ]:
# The columns of a matrix are where the basis vectors land.
i_hat = np.array([1.0, 0.0])
j_hat = np.array([0.0, 1.0])

print('A =')
print(A)
print()
print('A @ i_hat =', A @ i_hat, ' <- exactly column 0 of A:', A[:, 0])
print('A @ j_hat =', A @ j_hat, ' <- exactly column 1 of A:', A[:, 1])
print()
print('That is the whole content of "a matrix is a transformation":')
print('four numbers record two destinations, and everything else follows.')

In [ ]:
# Watch a matrix transform a shape. An asymmetric "house" so flips are visible.
house = np.array([
    [0, 0], [2, 0], [2, 1.4], [1, 2.2], [0, 1.4], [0, 0],
])

transforms = {
    'identity':          np.array([[1, 0], [0, 1]]),
    'scale x2':          np.array([[2, 0], [0, 2]]),
    'rotate 90 degrees': np.array([[0, -1], [1, 0]]),
    'shear':             np.array([[1, 1], [0, 1]]),
    'reflect':           np.array([[-1, 0], [0, 1]]),
    'squash (det = 0)':  np.array([[1, 0], [0, 0]]),
}

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (name, M) in zip(axes.ravel(), transforms.items()):
    # X @ M.T applies M to every ROW of X -- the standard ML orientation.
    moved = house @ M.T
    ax.plot(house[:, 0], house[:, 1], '--', color='gray', lw=1.2, label='before')
    ax.fill(moved[:, 0], moved[:, 1], alpha=0.3, color='#2a78d6')
    ax.plot(moved[:, 0], moved[:, 1], color='#2a78d6', lw=2, label='after')
    ax.set_title(f'{name}\ndet = {np.linalg.det(M):.1f}', fontsize=10)
    ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5); ax.set_aspect('equal')
    ax.axhline(0, color='#c3c2b7', lw=0.8); ax.axvline(0, color='#c3c2b7', lw=0.8)
plt.tight_layout()
plt.show()

print('Note "squash": the house is flattened onto a line. det = 0, and the transformation')
print('cannot be undone -- every point on a vertical line landed in the same place.')

Note `house @ M.T` rather than `M @ house`. Because our data stores examples as **rows**, applying a
transformation to every row means multiplying on the right by the transpose. Getting this backwards
produces a scrambled picture rather than an error, so it is worth checking deliberately.

### Meet a shape error on purpose

Shape errors are the most common failure in numerical Python. Read this one properly.

In [ ]:
A_3x2 = np.zeros((3, 2))
x_3 = np.zeros(3)

print('A_3x2 shape:', A_3x2.shape, ' x_3 shape:', x_3.shape)
print()
print('A @ x needs A columns == x length. Here that is 2 vs 3, so:')
show_error(lambda: A_3x2 @ x_3)
print()
print('The fix is to check the shapes first, every time:')
x_2 = np.ones(2)
print('A_3x2 @ x_2 =', A_3x2 @ x_2, 'with shape', (A_3x2 @ x_2).shape)
print()
print('Rule: (m x n) @ (n,) -> (m,).  n numbers go in, m numbers come out.')

### Exercises 13–16

In [ ]:
# TODO 13: compute [[1, 2], [0, 3]] @ (4, 1) using NumPy.
matvec_13 = None
check('13 · matrix-vector product', matvec_13, [6, 3])

# TODO 14: write the matrix that DOUBLES the x-coordinate and leaves y alone.
double_x = None
if double_x is not None:
    check('14 · applied to (3, 1)', np.array(double_x) @ np.array([3, 1]), [6, 1])
    check('14 · applied to (-2, 5)', np.array(double_x) @ np.array([-2, 5]), [-4, 5])
else:
    print('--    14 · double_x: not attempted yet')

# TODO 15: a matrix sends i_hat to (0, 0) and j_hat to (1, 1). Write that matrix.
#          Remember: the columns ARE the destinations.
collapsing = None
if collapsing is not None:
    check('15 · sends i_hat to (0,0)', np.array(collapsing) @ np.array([1, 0]), [0, 0])
    check('15 · sends j_hat to (1,1)', np.array(collapsing) @ np.array([0, 1]), [1, 1])
    check('15 · its determinant is 0', np.linalg.det(np.array(collapsing, dtype=float)), 0.0, tol=1e-9)
else:
    print('--    15 · collapsing: not attempted yet')

# TODO 16: a layer maps 784 inputs to 128 outputs. Create a zero weight matrix of the right shape.
#          Rows match outputs, columns match inputs.
W = None
check_shape('16 · W', W, (128, 784))
if W is not None:
    check_shape('16 · W @ input works', np.asarray(W) @ np.zeros(784), (128,))

## 6 · Composition, order, and the identity

*Companion to [Part 6 · Composing transformations](index.html#matmul).*

`AB` means "do `B` first, then `A`". Implement the product, then prove to yourself that order
matters.

In [ ]:
def matmul_from_scratch(A, B):
    '''Entry (i, j) of AB is row i of A dotted with column j of B.'''
    if A.shape[1] != B.shape[0]:
        raise ValueError(f'inner dimensions must match: {A.shape} @ {B.shape}')
    m, n = A.shape[0], B.shape[1]
    out = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            out[i, j] = dot_from_scratch(A[i, :], B[:, j])
    return out


A = np.array([[1.0, 2.0], [0.0, 3.0]])
B = np.array([[0.0, 1.0], [4.0, 5.0]])

print('my matmul:'); print(matmul_from_scratch(A, B))
print('numpy    :'); print(A @ B)
print('match    :', np.allclose(matmul_from_scratch(A, B), A @ B))

In [ ]:
# Order matters. Rotate-then-shear is not shear-then-rotate.
R = np.array([[0.0, -1.0], [1.0, 0.0]])     # rotate 90 degrees
S = np.array([[1.0, 1.0], [0.0, 1.0]])      # shear

print('R @ S ='); print(R @ S)
print('S @ R ='); print(S @ R)
print('equal? ', np.allclose(R @ S, S @ R))
print()
start = np.array([1.0, 0.0])
print('Applied to (1, 0):')
print('  R @ S @ (1,0) =', R @ (S @ start), '  <- shear first (it leaves (1,0) alone), then rotate')
print('  S @ R @ (1,0) =', S @ (R @ start), '  <- rotate first (up to (0,1)), then shear pushes it sideways')

In [ ]:
# The rules that DO hold.
C = np.array([[2.0, 1.0], [1.0, 1.0]])
I = np.eye(2)

print('associative   A(BC) == (AB)C :', np.allclose(A @ (B @ C), (A @ B) @ C))
print('distributive  A(B+C) == AB+AC:', np.allclose(A @ (B + C), A @ B + A @ C))
print('identity      A @ I == A     :', np.allclose(A @ I, A))
print('commutative   AB == BA       :', np.allclose(A @ B, B @ A), ' <- the one that fails')
print()
print('transpose of a product REVERSES the order:')
print('  (AB).T == B.T @ A.T :', np.allclose((A @ B).T, B.T @ A.T))
print('  (AB).T == A.T @ B.T :', np.allclose((A @ B).T, A.T @ B.T), ' <- the classic wrong answer')

In [ ]:
# Two nonzero matrices whose product is zero -- which is why you cannot cancel.
keep_x = np.array([[1.0, 0.0], [0.0, 0.0]])     # throws away y
keep_y = np.array([[0.0, 0.0], [0.0, 1.0]])     # throws away x

print('keep_x @ keep_y ='); print(keep_x @ keep_y)
print('Neither matrix is zero, but the composition destroys everything.')
print()

# The diagonal-matrix trap: left multiplies scale ROWS, right multiplies scale COLUMNS.
M = np.array([[1.0, 2.0], [3.0, 4.0]])
d = np.array([10.0, 100.0])
print('M ='); print(M)
print('np.diag(d) @ M  (scales ROWS):');    print(np.diag(d) @ M)
print('M @ np.diag(d)  (scales COLUMNS):'); print(M @ np.diag(d))
print()
print('Both run. Both look plausible. Only one is what you meant.')
print('When scaling FEATURES (which live in columns), you want the second.')

In [ ]:
# Why activation functions exist: stacked linear layers collapse into one matrix.
W1 = rng(1).normal(size=(4, 6))     # 6 inputs -> 4 hidden
W2 = rng(2).normal(size=(3, 4))     # 4 hidden -> 3 outputs
x = rng(3).normal(size=6)

two_layers = W2 @ (W1 @ x)
one_matrix = (W2 @ W1) @ x

print('W1 shape        :', W1.shape)
print('W2 shape        :', W2.shape)
print('W2 @ W1 shape   :', (W2 @ W1).shape, ' <- a SINGLE layer, 6 inputs -> 3 outputs')
print()
print('two layers      :', two_layers)
print('one matrix      :', one_matrix)
print('identical?      :', np.allclose(two_layers, one_matrix))
print()
print('A hundred stacked linear layers would also collapse to one matrix.')
print('The non-linearity between layers is the ONLY reason depth buys anything (Module 29).')

In [ ]:
# One matrix product computes EVERY pairwise similarity at once -- this is attention's QK^T.
E = np.array([                    # four little "embeddings"
    [1.0, 0.0],                   # 0: about models
    [0.9, 0.2],                   # 1: also about models
    [0.0, 1.0],                   # 2: about recipes
    [0.2, 0.9],                   # 3: also about recipes
])
E_unit = E / np.linalg.norm(E, axis=1, keepdims=True)   # normalise each ROW to length 1

similarity = E_unit @ E_unit.T    # (4x2) @ (2x4) -> (4x4)

print('E_unit shape   :', E_unit.shape)
print('similarity     :', similarity.shape, '-- every pair, in one product')
print(similarity)
print()
print('The diagonal is 1.0 (every vector is identical to itself).')
print('Items 0 and 1 score high with each other and low with 2 and 3.')
print('This matrix is the dot-product core of self-attention.')
print('Transformers additionally use learned Q and K projections, scaling, masks, and softmax (Module 37).')

### Exercises 17–19

In [ ]:
# TODO 17: compute [[2,1],[1,3]] @ [[1,0],[2,1]] using NumPy.
product_17 = None
check('17 · matrix product', product_17, [[4, 1], [7, 3]])

# TODO 18: find the inverse shear -- a matrix P such that [[1,1],[0,1]] @ P == I.
#          Think: one shear pushes right, so the other must push left by the same amount.
P = None
if P is not None:
    check('18 · undoes the shear', np.array([[1, 1], [0, 1]]) @ np.array(P), [[1, 0], [0, 1]])
else:
    print('--    18 · P: not attempted yet')

# TODO 19: X has shape (1000, 20). You want to reduce it to 5 features with a matrix.
#          Create a zero matrix of the right shape, and compute the reduced result.
X_big = np.zeros((1000, 20))
reducer = None                 # what shape must this be?
reduced = None                 # X_big combined with reducer

check_shape('19 · reducer', reducer, (20, 5))
check_shape('19 · reduced', reduced, (1000, 5))

## 7 · Determinant, rank, and inverses

*Companion to [Part 7 · Determinant, rank, and the inverse](index.html#invertibility).*

Five ideas, one picture: did the transformation flatten anything?

In [ ]:
matrices = {
    'independent columns': np.array([[3.0, 1.0], [2.0, 4.0]]),
    'column 2 = 2 x column 1': np.array([[2.0, 4.0], [1.0, 2.0]]),
    'rotation (area preserved)': np.array([[0.0, -1.0], [1.0, 0.0]]),
    'reflection (flips)': np.array([[-1.0, 0.0], [0.0, 1.0]]),
    'all zeros': np.zeros((2, 2)),
}

print(f'{"matrix":<28}{"det":>9}{"rank":>6}  invertible?')
for name, M in matrices.items():
    determinant = np.linalg.det(M)
    rank = np.linalg.matrix_rank(M)
    print(f'{name:<28}{determinant:>9.2f}{rank:>6}  {"yes" if rank == 2 else "NO"}')
print()
print('Note the second row: not one zero entry, yet det = 0 and rank = 1.')
print('Column 2 is exactly twice column 1, so the plane is flattened onto a line.')

In [ ]:
# The inverse undoes the transformation -- when one exists.
A = np.array([[1.0, 2.0], [3.0, 4.0]])
A_inv = np.linalg.inv(A)

print('A ='); print(A)
print('A_inv ='); print(A_inv)
print('A @ A_inv ='); print(A @ A_inv)
print('is it the identity?', np.allclose(A @ A_inv, np.eye(2)))
print()

# And when one does not.
singular = np.array([[2.0, 4.0], [1.0, 2.0]])
print('Trying to invert a singular matrix:')
show_error(np.linalg.inv, singular)
print()
print('You now know exactly what that message means: the columns were dependent,')
print('a dimension was destroyed, and no rule could send the points back.')

In [ ]:
# "Nearly singular" is the dangerous case -- no error, but the answer is unreliable.
print(f'{"matrix":<34}{"det":>12}{"condition number":>20}')
for epsilon in [1.0, 0.1, 0.01, 0.0001, 0.000001]:
    M = np.array([[1.0, 1.0], [1.0, 1.0 + epsilon]])
    print(f'{"[[1,1],[1,1+" + format(epsilon, "g") + "]]":<34}'
          f'{np.linalg.det(M):>12.2e}{np.linalg.cond(M):>20.2e}')
print()
print('For this fixed-scale family, nearly parallel columns make the determinant approach zero')
print('while the CONDITION NUMBER explodes. Determinant size alone is not a universal')
print('diagnostic because it changes with units; a large condition number is the warning.')
print('It means tiny data changes can produce large answer changes (Module 13).')

In [ ]:
# Why you should use solve() rather than inv(). Compare accuracy on an awkward matrix.
size = 12
awkward = np.vander(np.linspace(1, 2, size))     # a notoriously ill-conditioned matrix
true_x = np.ones(size)
b = awkward @ true_x

via_inverse = np.linalg.inv(awkward) @ b
via_solve = np.linalg.solve(awkward, b)

print('condition number     :', f'{np.linalg.cond(awkward):.3e}')
print('error using inv(A)@b :', f'{np.linalg.norm(via_inverse - true_x):.3e}')
print('error using solve()  :', f'{np.linalg.norm(via_solve - true_x):.3e}')
print()
print('Both answers are approximate, but solve() is meaningfully closer to the truth --')
print('and it is faster. Use np.linalg.solve, not np.linalg.inv, essentially always.')

### Exercises 20–22

In [ ]:
# TODO 20: compute the determinant of [[3, 1], [2, 4]] using NumPy.
det_20 = None
check('20 · determinant', det_20, 10.0, tol=1e-9)

# TODO 21: build a 3x3 matrix of rank 1 in which NO entry is zero.
#          Hint: make every row a multiple of the same row, e.g. (1, 2, 3).
rank_one = None
if rank_one is not None:
    check('21 · rank is 1', np.linalg.matrix_rank(np.array(rank_one, dtype=float)), 1)
    check('21 · no zero entries', float(np.all(np.array(rank_one) != 0)), 1.0)
    check_shape('21 · shape', rank_one, (3, 3))
else:
    print('--    21 · rank_one: not attempted yet')

# TODO 22: a dataset has "price in dollars", "price in euros" (0.9 x dollars), and "area".
#          Build it, then report its rank. Is it 3?
dollars = np.array([100.0, 250.0, 310.0, 90.0])
area = np.array([50.0, 120.0, 160.0, 45.0])
euros = None            # 0.9 times dollars
X_multi = None          # a (4, 3) matrix with columns dollars, euros, area
rank_22 = None          # the rank of X_multi

check_shape('22 · X_multi', X_multi, (4, 3))
check('22 · rank is 2, not 3', rank_22, 2)

## 8 · Linear systems and least squares

*Companion to [Part 8 · Solving Ax = b, and least squares](index.html#solving).*

The payoff section. You are about to derive linear regression before you have been taught it.

In [ ]:
# 2x + y = 5,  x + 3y = 10  --  as Ax = b.
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([5.0, 10.0])

solution = np.linalg.solve(A, b)
print('solution (x, y) :', solution)
print('check A @ x == b:', np.allclose(A @ solution, b))
print()
print('The column picture: how much of each column reaches b?')
print(f'  {solution[0]:.0f} * column 0 = {solution[0] * A[:, 0]}')
print(f'  {solution[1]:.0f} * column 1 = {solution[1] * A[:, 1]}')
print(f'  their sum      = {solution[0] * A[:, 0] + solution[1] * A[:, 1]}  == b')

In [ ]:
# The three cases. Same matrix, different b, opposite outcomes.
singular = np.array([[1.0, 2.0], [2.0, 4.0]])
print('det =', np.linalg.det(singular), ' rank =', np.linalg.matrix_rank(singular))
print('The columns are (1,2) and (2,4) -- both on one line, so the column space IS that line.')
print()

for b_try, label in [(np.array([3.0, 7.0]), 'b is OFF the line'),
                     (np.array([3.0, 6.0]), 'b is ON the line (it is 3 x (1,2))')]:
    print(f'b = {b_try}   {label}')
    show_error(np.linalg.solve, singular, b_try)
    print()

print('Both fail the same way, because solve() insists on a UNIQUE answer.')
print('One case has no solution; the other has infinitely many. lstsq handles both:')
print('  b off the line ->', np.linalg.lstsq(singular, np.array([3.0, 7.0]), rcond=None)[0], '(the closest it can get)')
print('  b on the line  ->', np.linalg.lstsq(singular, np.array([3.0, 6.0]), rcond=None)[0], '(one of infinitely many)')

In [ ]:
# Least squares: five points, two unknowns, no exact solution.
x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_data = np.array([2.2, 2.8, 4.5, 4.8, 6.4])

# Build X with a column of 1s so the model can have an intercept.
# Without it, a matrix can only produce lines through the origin.
X = np.column_stack([x_data, np.ones(len(x_data))])
print('X ='); print(X)
print('shape:', X.shape, ' -- 5 equations, 2 unknowns. Overdetermined: no exact solution exists.')
print()

# Three routes to the same answer.
normal_equations = np.linalg.solve(X.T @ X, X.T @ y_data)
lstsq_answer = np.linalg.lstsq(X, y_data, rcond=None)[0]

print('normal equations solve(X.T @ X, X.T @ y) :', normal_equations)
print('np.linalg.lstsq                          :', lstsq_answer)
print('they agree                               :', np.allclose(normal_equations, lstsq_answer))
print()
print(f'fitted line: y = {lstsq_answer[0]:.4f} x + {lstsq_answer[1]:.4f}')

In [ ]:
# And the same answer again, from scikit-learn -- so you can see it is the same thing.
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(x_data.reshape(-1, 1), y_data)
print('sklearn slope    :', model.coef_[0])
print('sklearn intercept:', model.intercept_)
print('matches our lstsq:', np.allclose([model.coef_[0], model.intercept_], lstsq_answer))
print()
print('LinearRegression().fit() solves exactly the equation you just solved by hand.')

In [ ]:
# The defining property: the residual is orthogonal to every column of X.
w = lstsq_answer
predictions = X @ w
residual = y_data - predictions

print('predictions :', predictions)
print('residual    :', residual)
print('sum of squared residuals :', residual @ residual)
print()
print('X.T @ residual =', X.T @ residual, ' <- zero, to floating-point precision')
print()
print('That is the whole of least squares. If the residual leaned along any feature')
print('direction, you could move further that way and get closer -- so at the closest')
print('point it must be perpendicular to all of them.')
print()
print('The second entry being zero is why the residuals SUM to zero:', f'{residual.sum():.2e}')
print('It is the perpendicularity condition applied to the column of 1s.')

In [ ]:
# Both pictures of the same fit.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

grid = np.linspace(0, 6, 100)
ax1.plot(grid, w[0] * grid + w[1], color='#eb6834', lw=2, label='least-squares line')
ax1.scatter(x_data, y_data, s=70, color='#2a78d6', zorder=3, label='data')
for xi, yi, pi in zip(x_data, y_data, predictions):
    ax1.plot([xi, xi], [yi, pi], color='#e34948', lw=2)
ax1.set_title('Residuals: the vertical gaps we are minimising')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.legend()

ax2.bar(range(len(residual)), residual, color='#e34948')
ax2.axhline(0, color='#c3c2b7')
ax2.set_title(f'The residual vector (its length squared = {residual @ residual:.3f})')
ax2.set_xlabel('data point'); ax2.set_ylabel('residual')
plt.tight_layout()
plt.show()

### Exercises 23–26

In [ ]:
# TODO 23: solve 3x - y = 1, x + 2y = 12 using np.linalg.solve.
A_23 = None
b_23 = None
solution_23 = None
check('23 · solution', solution_23, [2.0, 5.0], tol=1e-9)

# TODO 24: fit a least-squares line to these three points using np.linalg.lstsq.
#          Remember the column of 1s for the intercept.
x_24 = np.array([1.0, 2.0, 3.0])
y_24 = np.array([2.0, 3.0, 5.0])
X_24 = None            # shape (3, 2): the x values, then a column of ones
w_24 = None            # (slope, intercept)

check_shape('24 · X_24', X_24, (3, 2))
check('24 · slope and intercept', w_24, [1.5, 1.0 / 3.0], tol=1e-6)

# TODO 25: compute the residual vector for your fit, and confirm it sums to about zero.
residual_25 = None
check('25 · residual', residual_25, [1.0 / 6.0, -1.0 / 3.0, 1.0 / 6.0], tol=1e-6)
if residual_25 is not None:
    check('25 · residuals sum to zero', float(np.sum(residual_25)), 0.0, tol=1e-9)

# TODO 26: confirm the residual is orthogonal to BOTH columns of X_24
#          by computing X_24.T @ residual_25. It should be the zero vector.
orthogonality_26 = None
check('26 · orthogonality', orthogonality_26, [0.0, 0.0], tol=1e-9)

## 9 · Eigenvectors, eigenvalues, and PCA

*Companion to [Part 9 · Eigenvectors, eigenvalues, and a first look at PCA](index.html#eigen).*

In [ ]:
# Av = lambda v: the matrix scales the vector without knocking it off its line.
A = np.array([[2.0, 1.0], [1.0, 2.0]])          # symmetric
values, vectors = np.linalg.eig(A)

print('eigenvalues :', values)
print('eigenvectors (as COLUMNS of this matrix):')
print(vectors)
print()
for i in range(len(values)):
    v = vectors[:, i]
    print(f'eigenvector {i}: {v}')
    print(f'  A @ v      = {A @ v}')
    print(f'  lambda * v = {values[i] * v}')
    print(f'  equal?       {np.allclose(A @ v, values[i] * v)}')
print()
print('Because A is symmetric, its eigenvectors are guaranteed perpendicular:')
print('  their dot product =', f'{vectors[:, 0] @ vectors[:, 1]:.2e}')

In [ ]:
# A rotation has NO real eigenvectors -- every vector is knocked off its line.
rotation = np.array([[0.0, -1.0], [1.0, 0.0]])
values, _ = np.linalg.eig(rotation)
print('eigenvalues of a 90-degree rotation:', values)
print('They are COMPLEX (note the j). That is not a failure -- complex eigenvalues')
print('are precisely how the algebra reports "this transformation is a rotation".')

In [ ]:
# In this diagonal example, repeated application aligns with the unique dominant eigenvector.
A = np.array([[2.0, 0.0], [0.0, 0.5]])       # eigenvalues 2 and 0.5, along the axes
v = np.array([1.0, 1.0])                     # equal parts of both

print(f'{"step":>5}{"vector":>26}{"direction":>26}')
current = v.copy()
for step in range(9):
    print(f'{step:>5}{str(np.round(current, 3)):>26}{str(np.round(current / np.linalg.norm(current), 3)):>26}')
    current = A @ current
print()
print('After 8 steps the first component has been multiplied by 2^8 = 256 and the second')
print('by 0.5^8 = 0.0039. The direction has converged to (1, 0) -- the dominant eigenvector.')
print('This convergence relies on one eigenvalue having strictly largest magnitude and')
print('the start having a nonzero component in its eigen-direction.')
print('This is the power-iteration idea behind PageRank.')
print('Deep networks are more general: their stability depends on products of layer')
print('Jacobians and their singular values, not one matrix eigenvalue (Module 31).')

In [ ]:
# Markov steady state -- Demo 15, in code.
leave, back = 0.10, 0.05
M = np.array([[1 - leave, back],
              [leave, 1 - back]])            # columns sum to 1: nobody is lost

state = np.array([0.9, 0.1])                 # 90% start in the city
print('year  city    country')
for year in range(0, 41):
    if year % 8 == 0:
        print(f'{year:>4}  {state[0]:.4f}  {state[1]:.4f}')
    state = M @ state

steady_predicted = back / (leave + back)
print()
print('converged city fraction  :', f'{state[0]:.6f}')
print('predicted steady state   :', f'{steady_predicted:.6f}   (= back / (leave + back))')
print()
values, vectors = np.linalg.eig(M)
index = int(np.argmin(np.abs(values - 1.0)))
steady_vector = np.real(vectors[:, index])
steady_vector = steady_vector / steady_vector.sum()      # scale so the parts sum to 1
print('eigenvalues of M         :', np.real(values))
print('eigenvector for lambda=1 :', steady_vector, ' <- the same answer')

In [ ]:
# PCA from scratch: centre, covariance, eigenvectors.
generator = rng(7)
n = 300
long_axis = generator.normal(0, 2.4, n)
short_axis = generator.normal(0, 0.6, n)
theta = np.radians(30)
rotation = np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])
cloud = np.column_stack([long_axis, short_axis]) @ rotation.T + np.array([4.0, 2.0])

# Step 1 -- CENTRE the data. Skip this and PC1 just points at the mean.
mean = cloud.mean(axis=0)
centred = cloud - mean

# Step 2 -- the covariance matrix. Symmetric by construction.
covariance = centred.T @ centred / (len(centred) - 1)
print('covariance ='); print(covariance)
print('symmetric?', np.allclose(covariance, covariance.T))
print('numpy agrees:', np.allclose(covariance, np.cov(centred, rowvar=False)))
print()

# Step 3 -- eigenvectors. Use eigh for symmetric matrices: faster and more accurate.
values, vectors = np.linalg.eigh(covariance)
order = np.argsort(values)[::-1]                 # largest variance first
values, vectors = values[order], vectors[:, order]

print('variance along PC1 :', f'{values[0]:.3f}')
print('variance along PC2 :', f'{values[1]:.3f}')
print('PC1 direction      :', np.round(vectors[:, 0], 3))
print('PC2 direction      :', np.round(vectors[:, 1], 3))
print('perpendicular?     :', f'{vectors[:, 0] @ vectors[:, 1]:.2e}')
print()
print('PC1 explains', f'{100 * values[0] / values.sum():.1f}%', 'of the total variance.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

ax1.scatter(cloud[:, 0], cloud[:, 1], s=14, alpha=0.5, color='#2a78d6')
for i, color in enumerate(['#eb6834', '#1baf7a']):
    direction = vectors[:, i] * np.sqrt(values[i]) * 2.5
    ax1.annotate('', xy=mean + direction, xytext=mean,
                 arrowprops=dict(arrowstyle='->', color=color, lw=3))
    ax1.text(*(mean + direction * 1.15), f'PC{i + 1}', color=color, fontweight='bold')
ax1.set_title('The cloud, with its principal directions'); ax1.set_aspect('equal')

# Project onto the principal components: the same data, described in a better basis.
projected = centred @ vectors
ax2.scatter(projected[:, 0], projected[:, 1], s=14, alpha=0.5, color='#1baf7a')
ax2.set_title('The same data, rotated into the PC basis')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2'); ax2.set_aspect('equal')
plt.tight_layout()
plt.show()

print('In the PC basis the cloud is axis-aligned: PC1 carries the spread, PC2 the rest.')
print('Keeping only the PC1 column would be dimensionality reduction, 2-D down to 1-D.')

### Exercises 27–29

In [ ]:
# TODO 27: verify by hand-ish arithmetic that (1, 1) is an eigenvector of [[2,1],[1,2]].
#          Compute A @ v and report the eigenvalue (the factor v was scaled by).
A_27 = np.array([[2.0, 1.0], [1.0, 2.0]])
v_27 = np.array([1.0, 1.0])
Av_27 = None            # A_27 @ v_27
eigenvalue_27 = None    # the number such that Av == eigenvalue * v

check('27 · A @ v', Av_27, [3.0, 3.0])
check('27 · eigenvalue', eigenvalue_27, 3.0)

# TODO 28: use np.linalg.eigh to get the eigenvalues of [[3, 0], [0, 1]],
#          then report the LARGEST one.
largest_28 = None
check('28 · largest eigenvalue', largest_28, 3.0)

# TODO 29: given these covariance eigenvalues, what fraction of the total variance
#          do the first TWO components capture? Answer as a number between 0 and 1.
eigenvalues_29 = np.array([8.0, 1.5, 0.4, 0.1])
fraction_29 = None
check('29 · variance explained by the top 2', fraction_29, 0.95, tol=1e-9)

## 10 · Mini-project · a classifier built from nothing but a dot product

*Companion to the [mini-project brief](index.html#reference).*

No training loop. No gradient descent. No model class. Just averaging vectors (Part 2) and one dot
product (Part 4) — and it works.

The dataset is the handwritten digits bundled with scikit-learn: 8×8 greyscale images, no download
required.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print('images shape :', digits.images.shape, ' <- 1797 pictures, each 8 x 8')
print('data shape   :', digits.data.shape, '  <- the same pictures FLATTENED to vectors of 64')
print('targets      :', np.unique(digits.target))
print()
print('Flattening is exactly Practice Lab A question 2: an 8x8 grid becomes a point')
print('in 64-dimensional space. You cannot draw it, but every formula still works.')

# Keep only the 0s and the 1s.
mask = (digits.target == 0) | (digits.target == 1)
X_all = digits.data[mask]
y_all = digits.target[mask]
print()
print('kept', X_all.shape[0], 'images of 0s and 1s; each is a vector of', X_all.shape[1], 'numbers')

In [ ]:
# Split into a training half and a test half, so we can measure honestly.
generator = rng(42)
shuffle = generator.permutation(len(X_all))
split = len(X_all) // 2
train_idx, test_idx = shuffle[:split], shuffle[split:]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]
print('train:', X_train.shape, ' test:', X_test.shape)

# THE ENTIRE MODEL: average each class, then subtract.
mean_zero = X_train[y_train == 0].mean(axis=0)     # a vector of 64 numbers
mean_one = X_train[y_train == 1].mean(axis=0)
w = mean_one - mean_zero                           # the weight vector -- our whole model

print()
print('mean_zero shape :', mean_zero.shape)
print('w shape         :', w.shape, ' <- 64 numbers. That is the model.')

In [ ]:
# Look at the model. It is a picture.
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, image, title in [
    (axes[0], mean_zero.reshape(8, 8), 'average 0'),
    (axes[1], mean_one.reshape(8, 8), 'average 1'),
    (axes[2], w.reshape(8, 8), 'w = mean(1) - mean(0)\nthe model itself'),
]:
    im = ax.imshow(image, cmap='RdBu_r' if 'w =' in title else 'gray_r')
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

print('The third panel is the weight vector rendered as an 8x8 image.')
print('Red pixels push the prediction toward 1; blue pixels push it toward 0.')
print('A linear model IS a template, and prediction asks how much your input looks like it.')

In [ ]:
# Choose the threshold: the midpoint between the two class averages, measured along w.
bias = -w @ (mean_zero + mean_one) / 2

def predict(images):
    '''Score = w . x + bias. Positive means "1", negative means "0".'''
    scores = images @ w + bias
    return (scores > 0).astype(int), scores

train_pred, _ = predict(X_train)
test_pred, test_scores = predict(X_test)

print('training accuracy :', f'{(train_pred == y_train).mean():.1%}')
print('test accuracy     :', f'{(test_pred == y_test).mean():.1%}')
print()
print('No gradient descent. No optimiser. No epochs. Two averages and one dot product.')

In [ ]:
# Look at the scores: the two classes separate cleanly along one direction.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test_scores[y_test == 0], bins=24, alpha=0.75, label='true 0s', color='#2a78d6')
ax.hist(test_scores[y_test == 1], bins=24, alpha=0.75, label='true 1s', color='#eb6834')
ax.axvline(0, color='#e34948', lw=2, ls='--', label='decision boundary')
ax.set_xlabel('score = w . x + bias'); ax.set_ylabel('count')
ax.set_title('One dot product per image is enough to separate these two classes')
ax.legend()
plt.show()

print('Everything left of the dashed line is classified 0; everything right is classified 1.')
print('That dashed line is a DECISION BOUNDARY, and in 64 dimensions it is a hyperplane')
print('perpendicular to w. You will meet it again in Module 16.')

### Part two · a matrix is a stack of patterns

Now the SVD from Part 9, on a real image. Every matrix can be written as a sum of simple layers,
sorted by importance — keep the first few and you have the best possible approximation using that
little information.

In [ ]:
# Build one big image by tiling digits, so there is real structure to compress.
tiles = digits.images[:64].reshape(8, 8, 8, 8).transpose(0, 2, 1, 3).reshape(64, 64)

U, singular_values, Vt = np.linalg.svd(tiles)
print('image shape     :', tiles.shape)
print('U               :', U.shape)
print('singular values :', singular_values.shape)
print('Vt              :', Vt.shape)
print()
print('first 8 singular values:', np.round(singular_values[:8], 1))
print('last 8 singular values :', np.round(singular_values[-8:], 4))
print()
print('They fall off fast. The small ones contribute almost nothing -- which is exactly')
print('what makes throwing them away a good deal.')

In [ ]:
def reconstruct(k):
    '''Keep only the k largest singular values -- the k most important layers.'''
    return (U[:, :k] * singular_values[:k]) @ Vt[:k, :]

fig, axes = plt.subplots(1, 5, figsize=(13, 3.2))
for ax, k in zip(axes, [1, 3, 8, 20, 64]):
    approximation = reconstruct(k)
    stored = k * (tiles.shape[0] + tiles.shape[1] + 1)
    ax.imshow(approximation, cmap='gray_r')
    ax.set_title(f'k = {k}\n{100 * stored / tiles.size:.0f}% of the numbers', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout()
plt.show()

for k in [1, 3, 8, 20]:
    kept = singular_values[:k].sum() / singular_values.sum()
    error = np.linalg.norm(tiles - reconstruct(k)) / np.linalg.norm(tiles)
    print(f'k = {k:>2}:  {kept:>6.1%} of the singular-value mass,  relative error {error:.3f}')
print()
print('Keeping the top few layers powers image compression, PCA, recommender systems,')
print('and topic modelling. It also motivates low-rank adaptation methods.')
print('LoRA learns a new low-rank weight update instead of truncating this matrix (Module 41).')

## 11 · Exit ticket

Four tasks. Do them without scrolling up. If you can complete all four, you are ready for Module 05.

In [ ]:
# TICKET 1 -- WRITE.
# Implement cosine similarity from scratch: no np.linalg.norm, no np.dot, no @.
# Use loops and arithmetic only. Raise ValueError for unequal lengths or a zero vector.
def cosine_from_scratch(a, b):
    pass   # replace with your implementation


if cosine_from_scratch(np.array([1.0, 0.0]), np.array([1.0, 1.0])) is not None:
    check('ticket 1 · 45 degrees', cosine_from_scratch(np.array([1.0, 0.0]), np.array([1.0, 1.0])),
          0.7071067811865475, tol=1e-6)
    check('ticket 1 · identical', cosine_from_scratch(np.array([2.0, 6.0]), np.array([20.0, 60.0])), 1.0, tol=1e-9)
    check('ticket 1 · orthogonal', cosine_from_scratch(np.array([1.0, 0.0]), np.array([0.0, 5.0])), 0.0, tol=1e-9)
else:
    print('--    ticket 1: not attempted yet')

In [ ]:
# TICKET 2 -- PREDICT.
# Before running, write your answers in the comments. Then run to check.
#
# A is (3, 4). B is (4, 2). x is a vector of length 4.
#   a) shape of A @ B      : ______
#   b) shape of A @ x      : ______
#   c) is B @ A legal?     : ______
#   d) shape of (A @ B).T  : ______

A = np.zeros((3, 4)); B = np.zeros((4, 2)); x = np.zeros(4)
print('a) A @ B     :', (A @ B).shape)
print('b) A @ x     :', (A @ x).shape)
print('c) B @ A     :', end=' ')
show_error(lambda: B @ A)
print('d) (A @ B).T :', (A @ B).T.shape)

In [ ]:
# TICKET 3 -- REPAIR.
# This function is meant to return the least-squares fit of y on x, as (slope, intercept).
# It contains TWO bugs. Find and fix them, then run the check.
def broken_fit(x, y):
    X = x.reshape(-1, 1)                       # bug 1 is on this line
    return np.linalg.inv(X) @ y                # bug 2 is on this line


x_check = np.array([1.0, 2.0, 3.0])
y_check = np.array([2.0, 3.0, 5.0])

print('Before fixing, it fails like this:')
show_error(broken_fit, x_check, y_check)
print()
print('Once fixed, this should PASS:')
try:
    check('ticket 3 · repaired fit', broken_fit(x_check, y_check), [1.5, 1.0 / 3.0], tol=1e-6)
except Exception as error:
    print(f'--    ticket 3: still raising {type(error).__name__}: {error}')

In [ ]:
# TICKET 4 -- EXPLAIN.
# Fill in the blanks in the string below, then run the cell and read your own answer aloud.
explanation = '''
A matrix with determinant zero is called ____________. Geometrically it ______________
the plane, which means its rank is less than ____ and its columns are linearly
____________. It has no inverse because ______________________________________.

Least squares is needed when Ax = b has ____ solution, which happens because b is
outside the ____________ space of A. The answer it returns is the ____________ of b
onto that space, and the residual is always ____________ to every column of A.
'''
print(explanation)

## 12 · Solutions

Use these to check your work, not to skip it. If one surprises you, go back to the lesson section it
came from before moving on.

In [ ]:
# --- Section 1 ---
vec = np.array([5, 0, -2, 7])
mat = np.array([[2, 5], [0, 1], [-3, 4]])
mat_t = mat.T
vec_column = vec.reshape(4, 1)                # or vec.reshape(-1, 1)

n_examples = songs.shape[0]
n_features = songs.shape[1]
value = songs[7, 2]                            # 8th song, 3rd feature: Python counts from 0
feature_means = songs.mean(axis=0)

# --- Section 2 ---
result_3 = 3 * np.array([2, -1]) - 2 * np.array([1, 4])          # -> [4, -11]
arrow_4 = np.array([4, 1]) - np.array([1, 5])                    # destination minus start -> [3, -4]

def combine(a, vec1, b, vec2):
    return a * vec1 + b * vec2

# --- Section 3 ---
norm_6 = norm_from_scratch(np.array([5, 12]))                    # -> 13.0

def distance(p, q):
    return np.linalg.norm(q - p)                                 # subtract first, then measure

def normalise(vec):
    length = np.linalg.norm(vec)
    return vec if length == 0 else vec / length                  # guard the zero vector

print('section 1-3 solutions loaded')

In [ ]:
# --- Section 4 ---
dot_9 = np.array([1, 2, 3]) @ np.array([4, 5, 6])                # -> 32
orthogonal_10 = np.array([-4, 3])                                # swap and negate: (a,b) -> (-b,a)
cos_11 = (np.array([1, 0]) @ np.array([1, 1])) / (np.linalg.norm([1, 0]) * np.linalg.norm([1, 1]))

def project(a, b):
    return ((a @ b) / (b @ b)) * b

# --- Section 5 ---
matvec_13 = np.array([[1, 2], [0, 3]]) @ np.array([4, 1])        # -> [6, 3]
double_x = np.array([[2, 0], [0, 1]])
collapsing = np.array([[0, 1], [0, 1]])                          # columns ARE the destinations
W = np.zeros((128, 784))                                         # rows = outputs, columns = inputs

# --- Section 6 ---
product_17 = np.array([[2, 1], [1, 3]]) @ np.array([[1, 0], [2, 1]])
P = np.array([[1, -1], [0, 1]])                                  # the opposite shear
reducer = np.zeros((20, 5))
reduced = X_big @ reducer

print('section 4-6 solutions loaded')

In [ ]:
# --- Section 7 ---
det_20 = np.linalg.det(np.array([[3.0, 1.0], [2.0, 4.0]]))       # -> 10.0
rank_one = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])           # every row a multiple of (1,2,3)
euros = 0.9 * dollars
X_multi = np.column_stack([dollars, euros, area])
rank_22 = np.linalg.matrix_rank(X_multi)                         # -> 2: euros adds nothing new

# --- Section 8 ---
A_23 = np.array([[3.0, -1.0], [1.0, 2.0]])
b_23 = np.array([1.0, 12.0])
solution_23 = np.linalg.solve(A_23, b_23)                        # -> [2, 5]

X_24 = np.column_stack([x_24, np.ones(3)])                       # the column of 1s is the intercept
w_24 = np.linalg.lstsq(X_24, y_24, rcond=None)[0]
residual_25 = y_24 - X_24 @ w_24
orthogonality_26 = X_24.T @ residual_25                          # -> [0, 0]

# --- Section 9 ---
Av_27 = A_27 @ v_27                                              # -> [3, 3]
eigenvalue_27 = 3.0
largest_28 = np.linalg.eigh(np.array([[3.0, 0.0], [0.0, 1.0]]))[0].max()
fraction_29 = eigenvalues_29[:2].sum() / eigenvalues_29.sum()    # -> 0.95

print('section 7-9 solutions loaded')

In [ ]:
# --- Exit ticket ---

# Ticket 1: cosine similarity with no NumPy helpers at all.
def cosine_from_scratch(a, b):
    if len(a) != len(b):
        raise ValueError('vectors must have equal length')
    dot = 0.0
    norm_a = 0.0
    norm_b = 0.0
    for ai, bi in zip(a, b):
        dot += ai * bi
        norm_a += ai * ai
        norm_b += bi * bi
    denominator = (norm_a ** 0.5) * (norm_b ** 0.5)
    if denominator == 0:
        raise ValueError('cosine similarity is undefined for a zero vector')
    return dot / denominator


# Ticket 2: (3,2) · (3,) · illegal, because B has 2 columns and A has 3 rows · (2,3)

# Ticket 3: two bugs.
#   bug 1 -- no column of ones, so the model could not have an intercept
#   bug 2 -- inv() needs a square matrix, and X is 3x2. Use lstsq (and never inv here anyway).
def fixed_fit(x, y):
    X = np.column_stack([x, np.ones(len(x))])
    return np.linalg.lstsq(X, y, rcond=None)[0]


check('ticket 1 · solution', cosine_from_scratch(np.array([1.0, 0.0]), np.array([1.0, 1.0])),
      0.7071067811865475, tol=1e-6)
check('ticket 3 · solution', fixed_fit(np.array([1.0, 2.0, 3.0]), np.array([2.0, 3.0, 5.0])),
      [1.5, 1.0 / 3.0], tol=1e-6)

# Ticket 4, filled in:
print('''
A matrix with determinant zero is called SINGULAR. Geometrically it FLATTENS (collapses)
the plane, which means its rank is less than 2 (its size) and its columns are linearly
DEPENDENT. It has no inverse because information was destroyed -- many different inputs
land on the same output, so nothing can send them back.

Least squares is needed when Ax = b has NO solution, which happens because b is
outside the COLUMN space of A. The answer it returns is the PROJECTION of b
onto that space, and the residual is always ORTHOGONAL (perpendicular) to every column of A.
''')

## Where you are now

You have implemented the dot product, the norm, matrix–vector multiplication, and matrix
multiplication from scratch and checked every one against NumPy. You have solved a linear system,
fitted a least-squares line three ways that agreed, found eigenvectors, run a PCA, compressed an
image with the SVD, and classified handwritten digits with a single dot product.

Return to [the lesson page](index.html#self-check) and take the fifteen-question self-check if you
have not already. Then go to [Module 05 · Calculus for ML](../05-calculus-for-ml/index.html), which
answers the question this module left open: what do you do when the model is *not* linear and there
is no formula for the best answer?